# 01 Import And Standardize Connect 4 Models

This notebook pulls the runnable saved models from the GitHub repo, standardizes them as full `.keras` models, and writes a manifest into `../models/`.

Why full saved models instead of weights only:
- weights by themselves are not enough for these bots because several models rely on different architectures and custom layers
- saving full models keeps the architecture + parameters together so the round robin notebook can load them reliably


In [1]:
import os
import sys
import warnings
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve().parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
from connect4_model_hub import import_and_standardize_models, MODELS_DIR

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


In [2]:
PREFER_DOWNLOAD = True
OVERWRITE = True

print(f"Project root: {ROOT}")
print(f"Models dir: {MODELS_DIR}")
print(f"Prefer GitHub download: {PREFER_DOWNLOAD}")
print(f"Overwrite standardized models: {OVERWRITE}")


Project root: /Users/zacgarland/r_projects/connect4-megabot/megabot
Models dir: /Users/zacgarland/r_projects/connect4-megabot/megabot/models
Prefer GitHub download: True
Overwrite standardized models: True


In [3]:
manifest = import_and_standardize_models(
    models_dir=MODELS_DIR,
    prefer_download=PREFER_DOWNLOAD,
    overwrite=OVERWRITE,
)
manifest


,name,source,source_url,role,notes,fetch_mode,original_format,standardized_format,standardized_path,input_shape
0,archie_resnet,GitHub sub-bots/archie,https://raw.githubusercontent.com/zac-garland/...,opponent_pool,ResNet-style CNN.,downloaded,h5,keras,/Users/zacgarland/r_projects/connect4-megabot/...,"[null, 6, 7, 2]"
1,archie_transformer,GitHub sub-bots/archie,https://raw.githubusercontent.com/zac-garland/...,opponent_pool,Transformer with custom positional layers.,downloaded,keras,keras,/Users/zacgarland/r_projects/connect4-megabot/...,"[null, 6, 7, 2]"
2,connor_cnn,GitHub sub-bots/connor,https://raw.githubusercontent.com/zac-garland/...,opponent_pool,Connor CNN model activated after GitHub update.,downloaded,h5,keras,/Users/zacgarland/r_projects/connect4-megabot/...,"[null, 6, 7, 1]"
3,dean_cnn,GitHub sub-bots/dean,https://raw.githubusercontent.com/zac-garland/...,m1_best_model,Current strongest benchmark bot. Use as M1.,downloaded,h5,keras,/Users/zacgarland/r_projects/connect4-megabot/...,"[null, 6, 7, 2]"
4,dean_transformer,GitHub sub-bots/dean,https://raw.githubusercontent.com/zac-garland/...,opponent_pool,Dean transformer checkpoint.,downloaded,keras,keras,/Users/zacgarland/r_projects/connect4-megabot/...,"[null, 42, 2]"
5,zac_cnn_best,GitHub sub-bots/zac,https://raw.githubusercontent.com/zac-garland/...,opponent_pool,Best validation checkpoint.,downloaded,keras,keras,/Users/zacgarland/r_projects/connect4-megabot/...,"[null, 6, 7, 2]"
6,zac_cnn_final,GitHub sub-bots/zac,https://raw.githubusercontent.com/zac-garland/...,opponent_pool,Final saved CNN model.,downloaded,keras,keras,/Users/zacgarland/r_projects/connect4-megabot/...,"[null, 6, 7, 2]"


In [4]:
sorted(p.name for p in MODELS_DIR.glob("*.keras"))


['archie_resnet.keras',
 'archie_transformer.keras',
 'connor_cnn.keras',
 'dean_cnn.keras',
 'dean_transformer.keras',
 'zac_cnn_best.keras',
 'zac_cnn_final.keras']

In [5]:
# --- Optional: add locally-trained models into megabot/models/ + manifest ---
# This keeps a clean separation: training outputs can live elsewhere, but round-robin
# uses the standardized inventory under MODELS_DIR.

import json
import shutil

REPO_ROOT = ROOT.parent
LOCAL_MODELS = [
    {
        "name": "zac_policy_dean_arch",
        "path": REPO_ROOT / "sub-bots" / "improve-weak-link" / "outputs" / "zac_policy_dean_arch.keras",
        "role": "opponent_pool",
        "source": "local sub-bots/improve-weak-link",
        "notes": "Dean DeepResNet_Connect4 arch trained on Zac X_train_final/y_train_final.",
    }
]

manifest_path = MODELS_DIR / "manifest.csv"
manifest = pd.read_csv(manifest_path)

for spec in LOCAL_MODELS:
    src = Path(spec["path"]).resolve()
    if not src.exists():
        raise FileNotFoundError(f"Local model not found: {src}")

    dst = (MODELS_DIR / f"{spec['name']}.keras").resolve()
    shutil.copy2(src, dst)

    # Load to confirm it is readable and capture input shape
    import tensorflow as tf

    m = tf.keras.models.load_model(dst, compile=False, safe_mode=False)
    input_shape = tuple(m.input_shape)

    row = {
        "name": spec["name"],
        "source": spec["source"],
        "source_url": "",
        "role": spec["role"],
        "notes": spec["notes"],
        "fetch_mode": "local_copy",
        "original_format": "keras",
        "standardized_format": "keras",
        "standardized_path": str(dst),
        "input_shape": json.dumps(list(input_shape)),
    }

    # Upsert by name
    manifest = manifest[manifest["name"] != spec["name"]]
    manifest = pd.concat([manifest, pd.DataFrame([row])], ignore_index=True)

manifest = manifest.sort_values("name").reset_index(drop=True)
manifest.to_csv(manifest_path, index=False)
print("Updated manifest with local models.")
manifest[manifest["name"].isin([spec["name"] for spec in LOCAL_MODELS])]


NameError: name 'json' is not defined